<a href="https://colab.research.google.com/github/riballes/mobile-rag-firewall-/blob/main/experiments/phi_ner/notebooks/05_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHI NER — Baseline Comparison

The strongest academic claim is "our approach outperforms existing alternatives." Without baselines, F1=0.99 is meaningless — maybe spaCy does 0.99 too.

**Systems compared:**

| System | Type | Size | What it proves |
|--------|------|------|---------------|
| Regex only | Rules | 0 MB | Lower bound — can't detect names/addresses |
| spaCy sm | Pre-trained NER | 12 MB | Generic small model |
| spaCy lg | Pre-trained NER | 560 MB | Does model size help without fine-tuning? |
| Presidio | PII detection | ~50 MB | Purpose-built Microsoft PII tool |
| BioClinicalBERT | Fine-tuned (clinical pre-training) | ~440 MB | Does clinical pre-training help? |
| Ours (DistilBERT) | Fine-tuned (general pre-training) | ~260 MB | Our contribution |

The **Pareto frontier** (F1 vs latency scatter) shows the accuracy/speed trade-off — the ideal model is top-left (high F1, low latency).

**Runtime**: GPU (T4) — needed for BioClinicalBERT fine-tuning

**Estimated time**: ~30-40 minutes

In [1]:
!pip install -q transformers torch wandb weave seqeval spacy presidio-analyzer presidio-anonymizer accelerate matplotlib numpy pandas
!python -m spacy download en_core_web_sm -q
!python -m spacy download en_core_web_lg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.5/983.5 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 161.4 MB/s eta 0:00:00
ERROR: pi

In [2]:
import wandb, weave, json, re, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset as TorchDataset
from transformers import (AutoModelForTokenClassification, AutoTokenizer, Trainer,
                          TrainingArguments, DataCollatorForTokenClassification, pipeline as hf_pipeline)
from seqeval.metrics import classification_report, f1_score
from seqeval.scheme import IOB2
from google.colab import userdata

try: wandb.login(key=userdata.get("WANDB_API_KEY"))
except: wandb.login()
weave.init("mobile-rag-firewall")
print(f"GPU: {torch.cuda.is_available()}")

LABEL_LIST = ["O", "B-NAME", "I-NAME", "B-ADDRESS", "I-ADDRESS"]
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: l for l, i in LABEL_TO_ID.items()}

train_data = weave.ref("phi-ner-train:latest").get().rows
val_data = weave.ref("phi-ner-val:latest").get().rows
test_data = weave.ref("phi-ner-test:latest").get().rows
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ricardo-morales-b to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
weave: Logged in as Weights & Biases user: ricardo-morales-b.
weave: View Weave data at https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/weave


GPU: True
Train: 8682, Val: 1861, Test: 1861


In [3]:
def evaluate_system(name, test_data, predict_fn):
    """Evaluate a NER system. predict_fn(tokens) -> list of BIO tags."""
    true_seqs, pred_seqs, latencies = [], [], []
    for ex in test_data:
        tokens, true_tags = list(ex["tokens"]), list(ex["ner_tags"])
        start = time.perf_counter()
        pred_tags = predict_fn(tokens)
        latencies.append(time.perf_counter() - start)
        # Ensure same length
        pred_tags = (pred_tags + ["O"] * len(tokens))[:len(tokens)]
        true_seqs.append(true_tags)
        pred_seqs.append(pred_tags)

    report = classification_report(true_seqs, pred_seqs, output_dict=True, mode='strict', scheme=IOB2, zero_division=0)
    f1_macro = f1_score(true_seqs, pred_seqs, mode='strict', scheme=IOB2, zero_division=0)
    lats = np.array(latencies)

    result = {
        "system": name, "f1_macro": f1_macro,
        "f1_NAME": report.get("NAME", {}).get("f1-score", 0),
        "f1_ADDRESS": report.get("ADDRESS", {}).get("f1-score", 0),
        "p_NAME": report.get("NAME", {}).get("precision", 0),
        "r_NAME": report.get("NAME", {}).get("recall", 0),
        "p_ADDRESS": report.get("ADDRESS", {}).get("precision", 0),
        "r_ADDRESS": report.get("ADDRESS", {}).get("recall", 0),
        "latency_p50": float(np.percentile(lats, 50) * 1000),
        "latency_p95": float(np.percentile(lats, 95) * 1000),
    }
    print(f"  {name}: F1={f1_macro:.4f} (NAME={result['f1_NAME']:.4f}, ADDR={result['f1_ADDRESS']:.4f})")
    return result

all_baselines = []
print("Evaluation function ready.")

Evaluation function ready.


## Baselines 1-4: Rule-based and Pre-trained Systems

These baselines require **no training** — they use existing models/rules as-is on our test data.

- **Regex only**: Applies SSN/DOB/phone/email patterns. Expected to score 0% for NAME and ADDRESS since regex can't detect free-form text entities. Establishes the lower bound.
- **spaCy sm/lg**: Maps spaCy's PERSON → NAME, GPE/LOC/FAC → ADDRESS. Tests whether generic NER models trained on web text can handle medical PHI.
- **Presidio**: Microsoft's purpose-built PII detection engine. Maps PERSON → NAME, LOCATION → ADDRESS. Tests whether a dedicated PII tool outperforms generic NER.

In [4]:
# Baseline 1: Regex only (cannot detect NAME or ADDRESS)
def predict_regex(tokens):
    return ["O"] * len(tokens)  # Regex can't do token-level NER for names/addresses
all_baselines.append(evaluate_system("Regex Only", test_data, predict_regex))

# Baseline 2: spaCy en_core_web_sm
import spacy
nlp_sm = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
SPACY_MAP = {"PERSON": "NAME", "GPE": "ADDRESS", "LOC": "ADDRESS", "FAC": "ADDRESS"}
def predict_spacy(tokens, nlp=nlp_sm):
    text = " ".join(tokens)
    doc = nlp(text)
    tags = ["O"] * len(tokens)
    for ent in doc.ents:
        etype = SPACY_MAP.get(ent.label_)
        if not etype: continue
        # Map char spans to token indices (approximate)
        for i, tok in enumerate(tokens):
            tok_start = text.index(tok, sum(len(t)+1 for t in tokens[:i]) if i > 0 else 0) if tok in text else -1
            if tok_start >= ent.start_char and tok_start < ent.end_char:
                tags[i] = f"B-{etype}" if tags[max(0,i-1)] == "O" or not tags[max(0,i-1)].endswith(etype) else f"I-{etype}"
    return tags
all_baselines.append(evaluate_system("spaCy sm", test_data, predict_spacy))

# Baseline 3: spaCy en_core_web_lg
nlp_lg = spacy.load("en_core_web_lg", disable=["parser", "lemmatizer"])
def predict_spacy_lg(tokens):
    return predict_spacy(tokens, nlp=nlp_lg)
all_baselines.append(evaluate_system("spaCy lg", test_data, predict_spacy_lg))

# Baseline 4: Presidio
from presidio_analyzer import AnalyzerEngine
analyzer = AnalyzerEngine()
PRESIDIO_MAP = {"PERSON": "NAME", "LOCATION": "ADDRESS"}
def predict_presidio(tokens):
    text = " ".join(tokens)
    results = analyzer.analyze(text=text, language="en")
    tags = ["O"] * len(tokens)
    for r in results:
        etype = PRESIDIO_MAP.get(r.entity_type)
        if not etype: continue
        for i, tok in enumerate(tokens):
            pos = sum(len(t)+1 for t in tokens[:i])
            if pos >= r.start and pos < r.end:
                tags[i] = f"B-{etype}" if i == 0 or tags[i-1] == "O" else f"I-{etype}"
    return tags
all_baselines.append(evaluate_system("Presidio", test_data, predict_presidio))

TypeError: Found input variables without list of list.

## Baseline 5: BioClinicalBERT (fine-tuned)

`emilyalsentzer/Bio_ClinicalBERT` is BERT pre-trained on MIMIC-III clinical notes — real medical text. Fine-tuning it on our data tests whether **domain-specific pre-training** improves PHI detection over vanilla BERT/DistilBERT.

If BioClinicalBERT significantly outperforms DistilBERT, it means clinical language understanding matters for PHI detection. If it doesn't, the task is learnable from patterns alone (Synthea names with digits, structured addresses) without deep medical knowledge.

Same training setup as notebook 01: 10 epochs, batch 16, lr 2e-5.

In [ ]:
class PHINERDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data, self.tokenizer, self.max_length = data, tokenizer, max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        tokens, ner_tags = self.data[idx]["tokens"], self.data[idx]["ner_tags"]
        enc = self.tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=self.max_length, padding=False)
        labels, prev = [], None
        for wid in enc.word_ids():
            if wid is None: labels.append(-100)
            elif wid != prev: labels.append(LABEL_TO_ID.get(ner_tags[wid] if wid < len(ner_tags) else "O", 0))
            else:
                tag = ner_tags[wid] if wid < len(ner_tags) else "O"
                if tag.startswith("B-"): labels.append(LABEL_TO_ID.get("I-"+tag[2:], 0))
                elif tag.startswith("I-"): labels.append(LABEL_TO_ID.get(tag, 0))
                else: labels.append(-100)
            prev = wid
        enc["labels"] = labels
        return {k: torch.tensor(v) for k, v in enc.items()}

def compute_metrics_seq(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    true_s, pred_s = [], []
    for ps, ls in zip(preds, eval_pred.label_ids):
        t, p = [], []
        for pr, la in zip(ps, ls):
            if la != -100: t.append(ID_TO_LABEL[la]); p.append(ID_TO_LABEL[pr])
        true_s.append(t); pred_s.append(p)
    return {"f1_entity": f1_score(true_s, pred_s, mode='strict', scheme=IOB2)}

# Fine-tune BioClinicalBERT
from transformers import EarlyStoppingCallback

BIO_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
wandb.init(project="mobile-rag-firewall", name="baseline-bioclinicalbert", tags=["baseline"], reinit=True)
tok = AutoTokenizer.from_pretrained(BIO_MODEL)
model = AutoModelForTokenClassification.from_pretrained(BIO_MODEL, num_labels=5, id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)
args = TrainingArguments(output_dir="models/bioclinicalbert", report_to="wandb", num_train_epochs=5,
    per_device_train_batch_size=24, learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_entity", save_total_limit=1, logging_steps=100,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2, dataloader_pin_memory=True)
trainer = Trainer(model=model, args=args, train_dataset=PHINERDataset(list(train_data), tok),
                  eval_dataset=PHINERDataset(list(val_data), tok),
                  data_collator=DataCollatorForTokenClassification(tok), compute_metrics=compute_metrics_seq,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
trainer.save_model("models/bioclinicalbert/best"); tok.save_pretrained("models/bioclinicalbert/best")
wandb.finish()

# Evaluate BioClinicalBERT
bio_ner = hf_pipeline("ner", model="models/bioclinicalbert/best", tokenizer="models/bioclinicalbert/best",
                       aggregation_strategy="simple", truncation=True)
def predict_bioclinical(tokens):
    text = " ".join(tokens)
    preds = bio_ner(text)
    tags = ["O"] * len(tokens)
    for p in preds:
        if p["entity_group"] in ("NAME", "ADDRESS"):
            for i, tok in enumerate(tokens):
                pos = sum(len(t)+1 for t in tokens[:i])
                if pos >= p["start"] and pos < p["end"]:
                    tags[i] = f"B-{p['entity_group']}" if i == 0 or tags[i-1] == "O" else f"I-{p['entity_group']}"
    return tags
all_baselines.append(evaluate_system("BioClinicalBERT", test_data, predict_bioclinical))

## Baseline 6: Our Fine-tuned DistilBERT + Comparison

Load our best model from the W&B artifact (`phi-ner-model:latest`) and evaluate on the same test set as all baselines.

The final comparison includes:
- **Table**: P/R/F1 per entity type, macro F1, latency P50/P95 for all 6 systems
- **Grouped bar chart**: F1 per entity type — visually shows where each system excels/fails
- **Pareto frontier scatter**: F1 vs latency — identifies the best accuracy/speed trade-off

Results published to Weave as `phi-ner-baseline-comparison` for dashboard tracking.

In [ ]:
# Our fine-tuned model from W&B artifact
try:
    run = wandb.init(project="mobile-rag-firewall", job_type="pull-model", reinit=True)
    artifact = run.use_artifact("phi-ner-model:latest")
    model_path = artifact.download(root="ner_model_cache")
    run.finish()
    print("Loaded model from W&B artifact")
except:
    model_path = "models/distilbert/best"
    print(f"Using local model: {model_path}")

our_ner = hf_pipeline("ner", model=model_path, tokenizer=model_path, aggregation_strategy="simple", truncation=True)
def predict_ours(tokens):
    text = " ".join(tokens)
    preds = our_ner(text)
    tags = ["O"] * len(tokens)
    for p in preds:
        if p["entity_group"] in ("NAME", "ADDRESS"):
            for i, tok in enumerate(tokens):
                pos = sum(len(t)+1 for t in tokens[:i])
                if pos >= p["start"] and pos < p["end"]:
                    tags[i] = f"B-{p['entity_group']}" if i == 0 or tags[i-1] == "O" else f"I-{p['entity_group']}"
    return tags
all_baselines.append(evaluate_system("Ours (DistilBERT)", test_data, predict_ours))

# Comparison table
print(f"\n{'='*80}\n  BASELINE COMPARISON\n{'='*80}")
print(f"  {'System':<20} {'F1 Macro':>9} {'F1 NAME':>9} {'F1 ADDR':>9} {'P50 ms':>8} {'P95 ms':>8}")
print(f"  {'-'*20} {'-'*9} {'-'*9} {'-'*9} {'-'*8} {'-'*8}")
for r in sorted(all_baselines, key=lambda x: -x["f1_macro"]):
    print(f"  {r['system']:<20} {r['f1_macro']:>9.4f} {r['f1_NAME']:>9.4f} {r['f1_ADDRESS']:>9.4f} {r['latency_p50']:>8.1f} {r['latency_p95']:>8.1f}")

# Bar chart
df = pd.DataFrame(all_baselines).set_index("system")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
df[["f1_NAME", "f1_ADDRESS"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("F1 per Entity Type"); axes[0].set_ylabel("F1"); axes[0].set_ylim(0, 1.05)
axes[0].legend(["NAME", "ADDRESS"]); axes[0].tick_params(axis='x', rotation=45)

# F1 vs latency scatter
for _, r in df.iterrows():
    axes[1].scatter(r["latency_p50"], r["f1_macro"], s=100)
    axes[1].annotate(r.name if hasattr(r, 'name') else "", (r["latency_p50"]+0.5, r["f1_macro"]))
axes[1].set_xlabel("Latency P50 (ms)"); axes[1].set_ylabel("F1 Macro")
axes[1].set_title("F1 vs Latency (Pareto Frontier)")
plt.tight_layout(); plt.show()

# Publish to Weave
weave.publish(all_baselines, name="phi-ner-baseline-comparison")
print("Published baseline comparison to Weave.")